In [ ]:
import numpy as np
import pandas as pd

: 

In [2]:
# Loading the dataset
customer_data = pd.read_csv(r"C:\Users\Nikhi\OneDrive\Desktop\E-commerce_Recommendation\Data\raw\olist_customers_dataset.csv")
order_data = pd.read_csv(r"C:\Users\Nikhi\OneDrive\Desktop\E-commerce_Recommendation\Data\raw\olist_orders_dataset.csv")
product_data = pd.read_csv(r"C:\Users\Nikhi\OneDrive\Desktop\E-commerce_Recommendation\Data\raw\olist_products_dataset.csv")
order_items_data = pd.read_csv(r"C:\Users\Nikhi\OneDrive\Desktop\E-commerce_Recommendation\Data\raw\olist_order_items_dataset.csv")
order_payments_data = pd.read_csv(r"C:\Users\Nikhi\OneDrive\Desktop\E-commerce_Recommendation\Data\raw\olist_order_payments_dataset.csv")
order_review_data = pd.read_csv(r"C:\Users\Nikhi\OneDrive\Desktop\E-commerce_Recommendation\Data\raw\olist_order_reviews_dataset.csv")
geolocation_data = pd.read_csv(r"C:\Users\Nikhi\OneDrive\Desktop\E-commerce_Recommendation\Data\raw\olist_geolocation_dataset.csv")
product_name = pd.read_csv(r"C:\Users\Nikhi\OneDrive\Desktop\E-commerce_Recommendation\Data\raw\product_category_name_translation.csv")
sellers_data = pd.read_csv(r"C:\Users\Nikhi\OneDrive\Desktop\E-commerce_Recommendation\Data\raw\olist_sellers_dataset.csv")

In [3]:
print("customer: ",customer_data.shape)
print("order: ",order_data.shape) 
print("product: ",product_data.shape)
print("order items: ",order_items_data.shape)
print("order payment: ",order_payments_data.shape)
print("order review: ",order_review_data.shape)
print("geolocation: ",geolocation_data.shape)
print("product name: ",product_name.shape)
print("sellers: ",sellers_data.shape)

customer:  (99441, 5)
order:  (99441, 8)
product:  (32951, 9)
order items:  (112650, 7)
order payment:  (103886, 5)
order review:  (99224, 7)
geolocation:  (1000163, 5)
product name:  (71, 2)
sellers:  (3095, 4)


### customer data and order data

In [4]:
customer_data.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [5]:
order_data.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [6]:
# Only delivered orders represent completed user-product interactions.
order_data = order_data[order_data.order_status == "delivered"]
# I filtered only delivered orders to ensure interactions reflect true user intent

In [7]:
order_payments_data.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [8]:
order_payments_data['order_id'].duplicated().sum()

np.int64(4446)

in order payments data contain dublicate orderid.
Why duplicates happen
Multiple payment records per order,
Multiple items per order

In [9]:
#aggregate payment value per order.
payment_agg = order_payments_data.groupby("order_id")["payment_value"].sum().reset_index()     #This avoids underestimating order value.

In [10]:
payment_agg.head()

,order_id,payment_value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


In [11]:
product_data.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [12]:
product_name.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [13]:
product_data = product_data.merge(product_name, on='product_category_name', how='left')
product_data.head(2)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art


In [14]:
order_review_data.head(2)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13


In [15]:
sellers_data.head(2)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP


In [16]:
order_items_data.head(2)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93


JOINING TABLE 

In [17]:
final_data = (
  order_data
  .merge(order_items_data, on="order_id")
  .merge(product_data, on="product_id")
  .merge(customer_data, on="customer_id")
  .merge(order_review_data, on="order_id", how="left")
  .merge(payment_agg, on="order_id")
)

In [18]:
final_data.head(2)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,customer_zip_code_prefix,customer_city,customer_state,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,3149,sao paulo,SP,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,...,47813,barreiras,BA,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,141.46


In [19]:
# Rename the column
final_data.rename(columns={
  "customer_unique_id": "user_id",
  "order_purchase_timestamp":"order_time",
  "product_category_name_english": "category"
},inplace=True)

In [20]:
print("Shape: ",final_data.shape)
final_data.head(2)

Shape:  (110837, 34)


,order_id,customer_id,order_status,order_time,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,customer_zip_code_prefix,customer_city,customer_state,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,3149,sao paulo,SP,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,...,47813,barreiras,BA,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,141.46


In [21]:
final_data.columns

Index(['order_id', 'customer_id', 'order_status', 'order_time',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date',
       'price', 'freight_value', 'product_category_name',
       'product_name_lenght', 'product_description_lenght',
       'product_photos_qty', 'product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm', 'category', 'user_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'review_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp', 'payment_value'],
      dtype='str')

In [22]:
final_data.duplicated().sum()

np.int64(0)

In [23]:
print(final_data.isnull().sum())

order_id                             0
customer_id                          0
order_status                         0
order_time                           0
order_approved_at                   15
order_delivered_carrier_date         2
order_delivered_customer_date        8
order_estimated_delivery_date        0
order_item_id                        0
product_id                           0
seller_id                            0
shipping_limit_date                  0
price                                0
freight_value                        0
product_category_name             1545
product_name_lenght               1545
product_description_lenght        1545
product_photos_qty                1545
product_weight_g                    18
product_length_cm                   18
product_height_cm                   18
product_width_cm                    18
category                          1567
user_id                              0
customer_zip_code_prefix             0
customer_city            

In [24]:
final_data.review_score.isnull().sum()

np.int64(827)

#### handling null values
Reviews: fill with neutral assumption, 
Categories: mark as “unknown”

In [25]:
final_data["review_score"] = final_data["review_score"].fillna(3)
final_data['category'] = final_data['category'].fillna('Unknown')

In [26]:
final_data.columns

Index(['order_id', 'customer_id', 'order_status', 'order_time',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date',
       'price', 'freight_value', 'product_category_name',
       'product_name_lenght', 'product_description_lenght',
       'product_photos_qty', 'product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm', 'category', 'user_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'review_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp', 'payment_value'],
      dtype='str')

In [27]:
# convert the datetime format to all datetime releated columns
date_col = ['order_time','order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date','review_creation_date']
for col in date_col:
  final_data[col] = pd.to_datetime(final_data[col])

In [28]:
final_data.describe().T

,count,mean,min,25%,50%,75%,max,std
order_time,110837,2018-01-01 17:56:18.321995,2016-10-03 09:44:50,2017-09-14 16:38:24,2018-01-20 17:37:33,2018-05-04 23:18:55,2018-08-29 15:00:37,NaN
order_approved_at,110822,2018-01-02 05:30:12.120138,2016-10-04 09:43:32,2017-09-15 02:15:14.250000,2018-01-22 13:45:02.500000,2018-05-05 12:15:18.500000,2018-08-29 15:10:26,NaN
order_delivered_carrier_date,110835,2018-01-05 00:50:49.208174,2016-10-08 10:34:01,2017-09-18 20:26:09,2018-01-24 15:54:39,2018-05-08 13:16:30,2018-09-11 19:48:28,NaN
order_delivered_customer_date,110829,2018-01-14 05:11:00.819388,2016-10-11 13:46:32,2017-09-26 17:57:28,2018-02-02 17:32:36,2018-05-15 17:56:42,2018-10-17 13:22:46,NaN
order_estimated_delivery_date,110837,2018-01-25 13:29:26.626668,2016-10-27 00:00:00,2017-10-05 00:00:00,2018-02-16 00:00:00,2018-05-28 00:00:00,2018-10-25 00:00:00,NaN
order_item_id,110837.0,1.198887,1.0,1.0,1.0,1.0,21.0,0.708635
price,110837.0,119.812893,0.85,39.9,74.9,133.9,6735.0,181.958188
freight_value,110837.0,19.93757,0.0,13.08,16.25,21.15,409.68,15.674301
product_name_lenght,109292.0,48.8108,5.0,42.0,52.0,57.0,76.0,10.007222
product_description_lenght,109292.0,786.389205,4.0,347.0,601.0,985.0,3992.0,650.981935


In [29]:
final_data2 = final_data.copy()

In [30]:
final_data2.head()

,order_id,customer_id,order_status,order_time,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,customer_zip_code_prefix,customer_city,customer_state,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,87285b34884572647811a353c7ac498a,...,3149,sao paulo,SP,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,595fac2a385ac33a80bd5114aec74eb8,...,47813,barreiras,BA,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,aa4383b373c6aca5d8797843e5594415,...,75265,vianopolis,GO,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18,2018-08-22 19:07:58,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1,d0b61bfb1de832b15ba9d266ca96e5b0,...,59296,sao goncalo do amarante,RN,359d03e676b3c069f62cadba8dd3f6e8,5.0,NaN,O produto foi exatamente o que eu esperava e e...,2017-12-03,2017-12-05 19:21:58,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1,65266b2da20d04dbe00c5c2d3bb7859e,...,9195,santo andre,SP,e50934924e227544ba8246aeb3770dd4,5.0,NaN,NaN,2018-02-17,2018-02-18 13:02:51,28.62


In [31]:
final_data2.columns

Index(['order_id', 'customer_id', 'order_status', 'order_time',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date',
       'price', 'freight_value', 'product_category_name',
       'product_name_lenght', 'product_description_lenght',
       'product_photos_qty', 'product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm', 'category', 'user_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'review_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp', 'payment_value'],
      dtype='str')

In [32]:
final_data2 = final_data2.drop(
  columns=['customer_id', 'order_status', 'order_approved_at', 'order_delivered_carrier_date',
          'order_delivered_customer_date', 'order_estimated_delivery_date','shipping_limit_date','seller_id',
          'price', 'freight_value', 'product_category_name','product_name_lenght', 'product_description_lenght',
          'product_photos_qty', 'customer_zip_code_prefix', 'review_id',  'review_comment_title','review_comment_message', 
          'review_creation_date', 'review_answer_timestamp', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'
  ]
)

In [33]:
final_data2.head()

,order_id,order_time,order_item_id,product_id,category,user_id,customer_city,customer_state,review_score,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,1,87285b34884572647811a353c7ac498a,housewares,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,4.0,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,1,595fac2a385ac33a80bd5114aec74eb8,perfumery,af07308b275d755c9edb36a90c618231,barreiras,BA,4.0,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,1,aa4383b373c6aca5d8797843e5594415,auto,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,5.0,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,1,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,5.0,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,1,65266b2da20d04dbe00c5c2d3bb7859e,stationery,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,5.0,28.62


In [34]:
final_data2.isnull().sum()

order_id          0
order_time        0
order_item_id     0
product_id        0
category          0
user_id           0
customer_city     0
customer_state    0
review_score      0
payment_value     0
dtype: int64

In [36]:
final_data2.duplicated().sum()

np.int64(391)

In [37]:
final_data2 = final_data2.drop_duplicates()

In [38]:
final_data2.dtypes

order_id                     str
order_time        datetime64[us]
order_item_id              int64
product_id                   str
category                     str
user_id                      str
customer_city                str
customer_state               str
review_score             float64
payment_value            float64
dtype: object

### IMPLICIT FEEDBACK ENGINEERING
Olist dataset has no rating per product data, so we must infer preference from behaviour.
Transform raw interactions into a numerical preference signal that answers:
“How strongly does user u like product p?”
This becomes the target signal for collaborative filtering models.


In [39]:
# 1. Define the base interaction feedback(purchase)
# assign base score to every purchase base score is 5 taken
purchase_weight = 5
final_data2['purchase_weight'] = purchase_weight

In [40]:
final_data2.head()

,order_id,order_time,order_item_id,product_id,category,user_id,customer_city,customer_state,review_score,payment_value,purchase_weight
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,1,87285b34884572647811a353c7ac498a,housewares,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,4.0,38.71,5
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,1,595fac2a385ac33a80bd5114aec74eb8,perfumery,af07308b275d755c9edb36a90c618231,barreiras,BA,4.0,141.46,5
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,1,aa4383b373c6aca5d8797843e5594415,auto,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,5.0,179.12,5
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,1,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,5.0,72.20,5
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,1,65266b2da20d04dbe00c5c2d3bb7859e,stationery,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,5.0,28.62,5


In [41]:
# 2.review based boost 
# assign review score where if user give review score more than 4 then the user get review weight 2 else 0

final_data2['review_weight'] = np.where(final_data2['review_score']>=4, 2, 0)
final_data2.head()

,order_id,order_time,order_item_id,product_id,category,user_id,customer_city,customer_state,review_score,payment_value,purchase_weight,review_weight
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,1,87285b34884572647811a353c7ac498a,housewares,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,4.0,38.71,5,2
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,1,595fac2a385ac33a80bd5114aec74eb8,perfumery,af07308b275d755c9edb36a90c618231,barreiras,BA,4.0,141.46,5,2
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,1,aa4383b373c6aca5d8797843e5594415,auto,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,5.0,179.12,5,2
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,1,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,5.0,72.20,5,2
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,1,65266b2da20d04dbe00c5c2d3bb7859e,stationery,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,5.0,28.62,5,2


In [42]:
# 3. payment value boost
# assign value weight where which user payment value have more than average then user will get 1 weight else 0
avg = np.mean(final_data2['payment_value'])
print(f'Average value:{avg:.2f}')
final_data2['value_weight'] = np.where(final_data2['payment_value']>avg, 1, 0)
final_data2.head()

Average value:179.47


,order_id,order_time,order_item_id,product_id,category,user_id,customer_city,customer_state,review_score,payment_value,purchase_weight,review_weight,value_weight
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,1,87285b34884572647811a353c7ac498a,housewares,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,4.0,38.71,5,2,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,1,595fac2a385ac33a80bd5114aec74eb8,perfumery,af07308b275d755c9edb36a90c618231,barreiras,BA,4.0,141.46,5,2,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,1,aa4383b373c6aca5d8797843e5594415,auto,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,5.0,179.12,5,2,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,1,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,5.0,72.20,5,2,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,1,65266b2da20d04dbe00c5c2d3bb7859e,stationery,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,5.0,28.62,5,2,0


In [43]:
# 4. Repeat purchase boost
# assign repeat weight 3 if the user will buy same product repeated
# get purchase counts where some of the user buy same product for multiple time 

In [44]:
purchase_counts = final_data2.groupby(['user_id','product_id']).size().reset_index(name="purchase_count")


In [48]:
purchase_counts['repeat_weight'] = np.where(purchase_counts['purchase_count']>1, 3, 0)
purchase_counts.head(5)

,user_id,product_id,purchase_count,repeat_weight
0,0000366f3b9a7992bf8c76cfdf3221e2,372645c7439f9661fbbacfd129aa92ec,1,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,5099f7000472b634fea8304448d20825,1,0
2,0000f46a3911fa3c0805444483337064,64b488de448a5324c4134ea39c28a34b,1,0
3,0000f6ccb0745a6a4b88665a16c9f078,2345a354a6f2033609bbf62bf5be9ef6,1,0
4,0004aac84e0df4da2b147fca70cf8255,c72e18b3fe2739b8d24ebf3102450f37,1,0


In [51]:
# merge the purchase_counts to the our final dataset
final_data2 = final_data2.merge(purchase_counts[['user_id', 'product_id', 'repeat_weight']], on=['user_id', 'product_id'], how='left')
final_data2.head()

,order_id,order_time,order_item_id,product_id,category,user_id,customer_city,customer_state,review_score,payment_value,purchase_weight,review_weight,value_weight,repeat_weight
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,1,87285b34884572647811a353c7ac498a,housewares,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,4.0,38.71,5,2,0,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,1,595fac2a385ac33a80bd5114aec74eb8,perfumery,af07308b275d755c9edb36a90c618231,barreiras,BA,4.0,141.46,5,2,0,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,1,aa4383b373c6aca5d8797843e5594415,auto,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,5.0,179.12,5,2,0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,1,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,5.0,72.20,5,2,0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,1,65266b2da20d04dbe00c5c2d3bb7859e,stationery,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,5.0,28.62,5,2,0,0


In [52]:
# final interaction score : combine all signals into one meaningfull score
final_data2['interaction_score'] = (final_data2.purchase_weight + final_data2.review_weight + final_data2.value_weight + final_data2.repeat_weight)
final_data2.head()

,order_id,order_time,order_item_id,product_id,category,user_id,customer_city,customer_state,review_score,payment_value,purchase_weight,review_weight,value_weight,repeat_weight,interaction_score
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,1,87285b34884572647811a353c7ac498a,housewares,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,4.0,38.71,5,2,0,0,7
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,1,595fac2a385ac33a80bd5114aec74eb8,perfumery,af07308b275d755c9edb36a90c618231,barreiras,BA,4.0,141.46,5,2,0,0,7
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,1,aa4383b373c6aca5d8797843e5594415,auto,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,5.0,179.12,5,2,0,0,7
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,1,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,5.0,72.20,5,2,0,0,7
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,1,65266b2da20d04dbe00c5c2d3bb7859e,stationery,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,5.0,28.62,5,2,0,0,7


## phase 3 Feature engineering
we engineer three feature:
1) user feature
2) product feature
3) time-based feature

#### 1) User Feature
Total purchese per user, average order value, favorite categories, purchase frequency, 

In [53]:
final_data2.head(2)

,order_id,order_time,order_item_id,product_id,category,user_id,customer_city,customer_state,review_score,payment_value,purchase_weight,review_weight,value_weight,repeat_weight,interaction_score
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,1,87285b34884572647811a353c7ac498a,housewares,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,4.0,38.71,5,2,0,0,7
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,1,595fac2a385ac33a80bd5114aec74eb8,perfumery,af07308b275d755c9edb36a90c618231,barreiras,BA,4.0,141.46,5,2,0,0,7


In [54]:
# Total purchases per user 
total_purchases = final_data2.groupby("user_id").size().reset_index(name="total_purchases")
total_purchases.head()

,user_id,total_purchases
0,0000366f3b9a7992bf8c76cfdf3221e2,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1
2,0000f46a3911fa3c0805444483337064,1
3,0000f6ccb0745a6a4b88665a16c9f078,1
4,0004aac84e0df4da2b147fca70cf8255,1


In [57]:
# Average order value 
user_aov = final_data2.groupby('user_id')['payment_value'].mean().reset_index(name="avg_order_value")
user_aov.head()

,user_id,avg_order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19
2,0000f46a3911fa3c0805444483337064,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,43.62
4,0004aac84e0df4da2b147fca70cf8255,196.89


In [58]:
# favorite catgories
# creating group with userid, category and column count how many times a customer bought from each category. And sort for each customer highest count comes first
fav_catgories = final_data2.groupby(['user_id',"category"]).size().reset_index(name="count").sort_values(['user_id', 'count'], ascending=False)  
# renameing the category to favorite category 
fav_catgories = fav_catgories.rename(columns={'category': 'favorite_category'})

In [59]:
# drop the duplicate values it keeps the first row and drop remaining rows (keep only the top category oer customer)
fav_catgories.drop_duplicates('user_id')

,user_id,favorite_category,count
95787,ffffd2657e2aad2907e67c3e9daecbeb,perfumery,1
95786,ffff5962728ec6157033ef9805bacc48,watches_gifts,1
95785,ffff371b4d645b6ecea244b27531430a,auto,1
95784,fffea47cd6d3cc0a88bd621562a9d061,baby,1
95783,fffcf5a5ff07b0908bd4e2dbc735a684,health_beauty,2
...,...,...,...
4,0004aac84e0df4da2b147fca70cf8255,telephony,1
3,0000f6ccb0745a6a4b88665a16c9f078,telephony,1
2,0000f46a3911fa3c0805444483337064,stationery,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,health_beauty,1


In [60]:
# sort the highest favorite category by count
fav_catgories.sort_values(['count'], ascending=False)

,user_id,favorite_category,count
75017,c8460e4251689ba205045f3ea17884a1,telephony,24
25949,4546caea018ad8c692964e3382debd19,health_beauty,21
39539,698e1cf81d01a3d389d96145f7fa6df8,auto,20
73419,c402f431464c72e27330a67f7b94d4fb,computers_accessories,20
5773,0f5ac8d5c31de21d2f25e24be15bbffb,furniture_decor,18
...,...,...,...
20,000ec5bff359e1c0ad76a81a45cb598f,home_appliances,1
19,000e309254ab1fc5ba99dd469d36bdb4,fashion_underwear_beach,1
90245,f108981245a86b509868e5fc57a3a85c,home_appliances,1
17,000d460961d6dbfa3ec6c9f5805769e1,telephony,1


In [61]:
# purchase frequency : How often user purchase (days between orders)
# Active Days = number of DISTINCT days on which the user was active (placed at least one order)
#active_days = count of unique order dates per user

purchase_freq = final_data2.groupby('user_id')['order_time'].agg(lambda x: (x.max() - x.min()).days +1).reset_index(name="active_days")
purchase_freq.sort_values('active_days', ascending=False)

,user_id,active_days
18609,32ea3bdedab835c3aa6cb68ce66565ef,634
74744,ccafc1c3f270410521c3c6f3b249870f,609
79220,d8f3c4f441a9b59a29f977df16724f38,583
54236,94e5ea5a8c1bf546db2739673060c43f,581
49498,87b3f231705783eb2217e25851c0a45d,573
...,...,...
93336,ffee94d548cef05b146d825a7648dab4,1
93337,ffeefd086fc667aaf6595c8fe3d22d54,1
93338,ffef0ffa736c7b3d9af741611089729b,1
93339,fff1afc79f6b5db1e235a4a6c30ceda7,1


In [62]:
user_feature = (total_purchases
  .merge(user_aov, on='user_id', how='left')
  .merge(fav_catgories, on='user_id', how='left')
  .merge(purchase_freq, on='user_id', how='left')
)
user_feature.head()

,user_id,total_purchases,avg_order_value,favorite_category,count,active_days
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,bed_bath_table,1,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,health_beauty,1,1
2,0000f46a3911fa3c0805444483337064,1,86.22,stationery,1,1
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,telephony,1,1
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,telephony,1,1


### Product feature 
product popularity, average product price, product review score, category encoding
1) product popularity : Total number of times product was purchased.
2) average product price : Mean selling price of a product.
3) product review score : Average review score per product.
4) category encoding : Product category text

In [63]:
final_data2.head()

,order_id,order_time,order_item_id,product_id,category,user_id,customer_city,customer_state,review_score,payment_value,purchase_weight,review_weight,value_weight,repeat_weight,interaction_score
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,1,87285b34884572647811a353c7ac498a,housewares,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,4.0,38.71,5,2,0,0,7
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,1,595fac2a385ac33a80bd5114aec74eb8,perfumery,af07308b275d755c9edb36a90c618231,barreiras,BA,4.0,141.46,5,2,0,0,7
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,1,aa4383b373c6aca5d8797843e5594415,auto,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,5.0,179.12,5,2,0,0,7
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,1,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,5.0,72.20,5,2,0,0,7
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,1,65266b2da20d04dbe00c5c2d3bb7859e,stationery,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,5.0,28.62,5,2,0,0,7


In [64]:
# product popularity
product_count = final_data2.groupby('product_id').size().reset_index(name="product_count")
product_count.head()

,product_id,product_count
0,00066f42aeeb9f3007548bb9d3f33c38,1
1,00088930e925c41fd95ebfe695fd2655,1
2,0009406fd7479715e4bef61dd91f2462,1
3,000b8f95fcb9e0096488278317764d19,2
4,000d9be29b5207b54e86aa1b1ac54872,1


In [65]:
# average product price
avg_product_price = final_data2.groupby('product_id')['payment_value'].mean().reset_index(name='avg_product_price')
avg_product_price.head()

,product_id,avg_product_price
0,00066f42aeeb9f3007548bb9d3f33c38,120.24
1,00088930e925c41fd95ebfe695fd2655,143.83
2,0009406fd7479715e4bef61dd91f2462,242.10
3,000b8f95fcb9e0096488278317764d19,78.50
4,000d9be29b5207b54e86aa1b1ac54872,218.27


In [66]:
# product review score
avg_review = final_data2.groupby('product_id')['review_score'].mean().reset_index(name='avg_product_review')
avg_review.head()

,product_id,avg_product_review
0,00066f42aeeb9f3007548bb9d3f33c38,5.0
1,00088930e925c41fd95ebfe695fd2655,4.0
2,0009406fd7479715e4bef61dd91f2462,1.0
3,000b8f95fcb9e0096488278317764d19,5.0
4,000d9be29b5207b54e86aa1b1ac54872,5.0


In [67]:
# category encoding
product_category = final_data2[['product_id','category']].drop_duplicates()
product_category.head()

,product_id,category
0,87285b34884572647811a353c7ac498a,housewares
1,595fac2a385ac33a80bd5114aec74eb8,perfumery
2,aa4383b373c6aca5d8797843e5594415,auto
3,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop
4,65266b2da20d04dbe00c5c2d3bb7859e,stationery


In [68]:
product_feature = (product_count
                   .merge(avg_product_price, on='product_id', how = 'left')
                   .merge(avg_review, on='product_id', how='left')
                   .merge(product_category, on='product_id', how='left')
                  )
product_feature.head()

,product_id,product_count,avg_product_price,avg_product_review,category
0,00066f42aeeb9f3007548bb9d3f33c38,1,120.24,5.0,perfumery
1,00088930e925c41fd95ebfe695fd2655,1,143.83,4.0,auto
2,0009406fd7479715e4bef61dd91f2462,1,242.10,1.0,bed_bath_table
3,000b8f95fcb9e0096488278317764d19,2,78.50,5.0,housewares
4,000d9be29b5207b54e86aa1b1ac54872,1,218.27,5.0,watches_gifts


### Time-Based feature
User preferences change over time.
Recent behavior > old behavior.
1) Recency: days since last purchase
2) Day of week: day when purchase occurred
3) Month: month of purchase

In [69]:
# Recency
last_purchase = (
    final_data2.groupby("user_id")["order_time"]
      .max()
      .reset_index(name="last_purchase_date")
)

analysis_date = final_data2['order_time'].max()
last_purchase["recency_days"] = (
    analysis_date - last_purchase["last_purchase_date"]
).dt.days


In [70]:
last_purchase.head()

,user_id,last_purchase_date,recency_days
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,111
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,114
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,536
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,320
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,287


In [71]:
# day of week
last_purchase['purchase_dayofweek'] = last_purchase['last_purchase_date'].dt.dayofweek
last_purchase.head()

,user_id,last_purchase_date,recency_days,purchase_dayofweek
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,111,3
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,114,0
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,536,4
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,320,3
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,287,1


In [72]:
# month of purchase
last_purchase['purchase_month'] = last_purchase['last_purchase_date'].dt.month
last_purchase.head()
time_feature = last_purchase.copy()
time_feature.head()

,user_id,last_purchase_date,recency_days,purchase_dayofweek,purchase_month
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,111,3,5
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,114,0,5
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,536,4,3
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,320,3,10
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,287,1,11


## Phase 4 : MODEL BUILDING 

### Model 1 : item based collaborative filtering
What it is? 
Recommends items similar to what a user already bought.
“Users who bought X also bought Y”

🔹 Why we use it

Works well with sparse data

Great for similar product sections

🔹 Data Used

User–Item interaction matrix

In [73]:
final_data2.head()

,order_id,order_time,order_item_id,product_id,category,user_id,customer_city,customer_state,review_score,payment_value,purchase_weight,review_weight,value_weight,repeat_weight,interaction_score
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,1,87285b34884572647811a353c7ac498a,housewares,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,4.0,38.71,5,2,0,0,7
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,1,595fac2a385ac33a80bd5114aec74eb8,perfumery,af07308b275d755c9edb36a90c618231,barreiras,BA,4.0,141.46,5,2,0,0,7
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,1,aa4383b373c6aca5d8797843e5594415,auto,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,5.0,179.12,5,2,0,0,7
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,1,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,5.0,72.20,5,2,0,0,7
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,1,65266b2da20d04dbe00c5c2d3bb7859e,stationery,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,5.0,28.62,5,2,0,0,7


In [74]:
user_feature.head()

,user_id,total_purchases,avg_order_value,favorite_category,count,active_days
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,bed_bath_table,1,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,health_beauty,1,1
2,0000f46a3911fa3c0805444483337064,1,86.22,stationery,1,1
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,telephony,1,1
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,telephony,1,1


In [75]:
# user_item_matrix = final_data2.pivot_table(index='user_id', columns='product_id', values='interaction_score', fill_value=0)
# user_item_matrix.head()
'''Error: MemoryError

Why this error happens:
Pandas is trying to create a very large matrix when you use pivot_table.
In your case, the shape is (32,216 users × 93,357 products), 
which means more than 3 billion cells. Since the data type is float64 (8 bytes per cell), 
it needs about 22.4 GB of RAM. Your system does not have that much free memory, 
so Python cannot allocate it and raises a MemoryError.
'''

'Error: MemoryError\n\nWhy this error happens:\nPandas is trying to create a very large matrix when you use pivot_table.\nIn your case, the shape is (32,216 users × 93,357 products), \nwhich means more than 3 billion cells. Since the data type is float64 (8 bytes per cell), \nit needs about 22.4 GB of RAM. Your system does not have that much free memory, \nso Python cannot allocate it and raises a MemoryError.\n'

In [67]:
from sklearn.preprocessing import LabelEncoder

user_encoder = LabelEncoder()
product_encoder = LabelEncoder()

final_data2['user_idx'] = user_encoder.fit_transform(final_data2['user_id'])
final_data2['product_idx'] = product_encoder.fit_transform(final_data2['product_id'])


In [68]:
from scipy.sparse import csr_matrix

# create user product matrix
user_product_matrix = csr_matrix(    
    (
        final_data2['interaction_score'],
        (final_data2['user_idx'], final_data2['product_idx'])
    )
)

In [69]:
# transpose: product × users
# Transpose for Item Similarity
product_user_matrix = user_product_matrix.T
product_user_matrix

<Compressed Sparse Column sparse matrix of dtype 'int64'
	with 99784 stored elements and shape (32216, 93357)>

In [70]:
# Compute Cosine Similarity
from sklearn.metrics.pairwise import cosine_similarity

# compute similarities only for top-K items
product_similarity = cosine_similarity(product_user_matrix)

In [71]:
# Recommendation Logic
def similar_products(product_id, top_k=10):
  if product_id not in product_encoder.classes_:
    return "Product not found (cold start)"
  
  idx = product_encoder.transform([product_id])[0]
  
  similarity_scores = product_similarity[idx].toarray().ravel()

  top_indices = np.argsort(similarity_scores)[::-1][1:top_k+1]

  return product_encoder.inverse_transform(top_indices)


### MODEL 2: MATRIX FACTORIZATION (IMPLICIT ALS)
🔹 What it is

Learns latent preferences between users and products.

🔹 Why ALS (not SVD)?
| Reason                 | Explanation                     |
| ---------------------- | ------------------------------- |
| Implicit data          | Optimized for confidence scores |
| Scales well            | Industry standard               |
| Strong personalization | Learns hidden patterns          |

🔹 Data Used

Sparse interaction matrix

In [72]:
# Encode IDs
user_ids = final_data2["user_id"].astype("category").cat.codes
item_ids = final_data2["product_id"].astype("category").cat.codes

In [73]:
# Create sparce matrix
interaction_matrix = csr_matrix(
  (final_data2['interaction_score'],(user_ids, item_ids))
)

In [74]:
interaction_matrix

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 99784 stored elements and shape (93357, 32216)>

In [75]:
# Train ALS model
from implicit.als import AlternatingLeastSquares

als_model = AlternatingLeastSquares(
  factors=100,
  regularization=0.1,
  iterations=20
)
als_model.fit(interaction_matrix)

c:\Users\Nikhi\OneDrive\Desktop\E-commerce_Recommendation\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Nikhi\OneDrive\Desktop\E-commerce_Recommendation\venv\lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 20/20 [00:12<00:00,  1.55it/s]


In [76]:
# def als_recommend(user_id, N=10):
#  if user_id not in in



In [77]:
als_model.recommend(user_ids, interaction_matrix[user_ids], N=10)

(array([[13749, 31097,  1340, ..., 17640, 31347, 26524],
        [ 1808, 20755, 29723, ..., 21971, 12074, 13466],
        [10589, 29536, 28610, ..., 25788, 21011,  5323],
        ...,
        [24746, 10503,  1287, ..., 20822, 18552,  3570],
        [24746, 10503,  1287, ..., 20822, 18552,  3570],
        [ 8429, 28610, 29536, ..., 25788, 21011,  5323]],
       shape=(110837, 10), dtype=int32),
 array([[1.06392659e-01, 9.89613980e-02, 9.67832059e-02, ...,
         8.99480581e-02, 8.81755203e-02, 8.54331106e-02],
        [2.16328442e-01, 1.87461525e-01, 1.19759664e-01, ...,
         9.38225761e-02, 9.27858278e-02, 9.10081789e-02],
        [1.60473137e-04, 1.07152104e-04, 1.06808948e-04, ...,
         9.68562599e-05, 9.35182034e-05, 9.20832899e-05],
        ...,
        [1.95911705e-01, 1.60961598e-01, 9.85194445e-02, ...,
         7.25861117e-02, 7.18870163e-02, 7.09487200e-02],
        [1.95911705e-01, 1.60961598e-01, 9.85194445e-02, ...,
         7.25861117e-02, 7.18870163e-02, 7.09487

### MODEL 3: CONTENT-BASED FILTERING
🔹 What it is

Recommends similar products based on metadata.

🔹 Why we use it

Handles new users & new products

Business-safe recommendations

🔹 Data Used

Product category, price bucket, popularity

In [78]:
product_feature.head()

,product_id,product_count,avg_product_price,avg_product_review,category
0,00066f42aeeb9f3007548bb9d3f33c38,1,120.24,5.0,perfumery
1,00088930e925c41fd95ebfe695fd2655,1,143.83,4.0,auto
2,0009406fd7479715e4bef61dd91f2462,1,242.10,1.0,bed_bath_table
3,000b8f95fcb9e0096488278317764d19,2,78.50,5.0,housewares
4,000d9be29b5207b54e86aa1b1ac54872,1,218.27,5.0,watches_gifts


In [79]:
def price_bucket(price):
  if price <=50:
    return "low-price"
  elif price <=200:
    return "mid-price"
  else: 
    return "high-price"
  
product_feature['price_bucket'] = product_feature['avg_product_price'].apply(price_bucket)

In [80]:
# build product text features
product_feature["text"] = (product_feature["category"] + " " +  product_feature['price_bucket'])

In [81]:
product_feature.head()

,product_id,product_count,avg_product_price,avg_product_review,category,price_bucket,text
0,00066f42aeeb9f3007548bb9d3f33c38,1,120.24,5.0,perfumery,mid-price,perfumery mid-price
1,00088930e925c41fd95ebfe695fd2655,1,143.83,4.0,auto,mid-price,auto mid-price
2,0009406fd7479715e4bef61dd91f2462,1,242.10,1.0,bed_bath_table,high-price,bed_bath_table high-price
3,000b8f95fcb9e0096488278317764d19,2,78.50,5.0,housewares,mid-price,housewares mid-price
4,000d9be29b5207b54e86aa1b1ac54872,1,218.27,5.0,watches_gifts,high-price,watches_gifts high-price


In [82]:
# TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
product_vectors = tfidf.fit_transform(product_feature['text'])

In [83]:
# similarity matrix
content_similarity = cosine_similarity(product_vectors)

In [84]:
# recommend similar products
def content_recommend(product_id, top_k=10):
    idx = product_feature.index[product_feature["product_id"] == product_id][0]
    scores = list(enumerate(content_similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return [product_feature.iloc[i[0]]["product_id"] for i in scores[1:top_k+1]]


### MODEL 4: POPULARITY + GEO MODEL
🔹 What it is

Most-purchased products globally or by state.

🔹 Why we use it

New users

No history

Safe fallback

In [85]:
final_data2.head()

,order_id,order_time,order_item_id,product_id,category,user_id,customer_city,customer_state,review_score,payment_value,purchase_weight,review_weight,value_weight,repeat_weight,interaction_score,user_idx,product_idx
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,1,87285b34884572647811a353c7ac498a,housewares,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,4.0,38.71,5,2,0,0,7,45377,17026
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,1,595fac2a385ac33a80bd5114aec74eb8,perfumery,af07308b275d755c9edb36a90c618231,barreiras,BA,4.0,141.46,5,2,0,0,7,63940,11335
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,1,aa4383b373c6aca5d8797843e5594415,auto,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,5.0,179.12,5,2,0,0,7,21375,21306
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,1,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,5.0,72.20,5,2,0,0,7,45325,26306
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,1,65266b2da20d04dbe00c5c2d3bb7859e,stationery,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,5.0,28.62,5,2,0,0,7,41825,12768


In [86]:
popular_products = (final_data2.groupby('product_id').value_counts().sort_values(ascending = False).head(10).index.tolist())

In [87]:
popular_products

[('3025303c80b01a0926fda2841cc31b4a',
  'df56136b8031ecd28e200bb18e6ddb2e',
  Timestamp('2017-01-26 13:15:41'),
  3,
  'sports_leisure',
  '2e43e031f10de28e557c35ef668f9396',
  'canoas',
  'RS',
  5.0,
  204.72,
  5,
  2,
  1,
  3,
  11,
  16935,
  6137),
 ('50aa8f292a9510d5542f2a078903a2a7',
  'df56136b8031ecd28e200bb18e6ddb2e',
  Timestamp('2017-01-26 13:15:41'),
  2,
  'sports_leisure',
  '2e43e031f10de28e557c35ef668f9396',
  'canoas',
  'RS',
  5.0,
  204.72,
  5,
  2,
  1,
  3,
  11,
  16935,
  10233),
 ('ac20a9614b6db9e7289b85c4f4b6216a',
  '8e17072ec97ce29f0e1f111e598b0c85',
  Timestamp('2018-03-31 15:08:21'),
  1,
  'home_appliances',
  '66980c3775537536f77b434d74e520f5',
  'belo horizonte',
  'MG',
  1.0,
  59.32,
  5,
  0,
  0,
  3,
  8,
  37523,
  21544),
 ('f28cbd414cd06f1ae84065c8dd8be834',
  'df56136b8031ecd28e200bb18e6ddb2e',
  Timestamp('2017-01-26 13:15:41'),
  1,
  'sports_leisure',
  '2e43e031f10de28e557c35ef668f9396',
  'canoas',
  'RS',
  5.0,
  204.72,
  5,
  2,
 

In [88]:
def recommend(user_id):
    if is_new_user(user_id):
        return geo_popular(user_id)

    if user_interactions(user_id) < 5:
        return content_based(user_id)

    return hybrid_rank(user_id)


Final Score =
0.5 × ALS +
0.3 × Item-CF +
0.2 × Content


final_score = (
    0.5 * als_score +
    0.3 * item_score +
    0.2 * content_score
)


## PHASE 5: SMART RECOMMENDATION LOGIC

(Decision Engine + Business Intelligence Layer)

🎯 WHY PHASE 5 IS CRITICAL

Without smart logic:

Models contradict each other

Cold-start users get bad results

Business rules are ignored

Recommendations feel random

STEP 1: USER CONTEXT EVALUATION

In [89]:
def get_user_context(user_id):
  user_data = final_data2[final_data2['user_id'] == user_id]

  return {
    "is_new": user_data.empty,
    "interaction_count": user_data.shape[0],
    "last_purchase_days": ( pd.Timestamp('now') - user_data["order_time"].max()).days if not user_data.empty else None,
    "state": user_data["customer_date"].iloc[0] if not user_data.empty else None
  }

STEP 2: model selection logic

In [90]:
def select_models(context):
  if context["is_new"]:
    return ["geo_popular"]
  
  if context["interaction_count"] < 5:
    return ["content"]
  
  return ["als", 'item_cf', "content"]


STEP 3: Candidate generation

condidate = a product that might be recommendaed

we generate more than needed, then rank

In [91]:
def generate_candidates(user_id, models):
  candidates = set()

  if "als" in models:
        candidates.update(als_recommend(user_id, n=50))

  if "item_cf" in models:
      candidates.update(item_cf_recommend(user_id, n=50))

  if "content" in models:
       candidates.update(content_based(user_id, n=50))

  return list(candidates)

STEP 4: HYBRID SCORING (CORE INTELLIGENCE)
❓ What happens here?

Each candidate product gets multiple scores:
| Model      | Score            |
| ---------- | ---------------- |
| ALS        | als_score        |
| Item-CF    | item_score       |
| Content    | content_score    |
| Popularity | popularity_score |
| Recency    | time_score       |


In [93]:
final_score =( 0.5 * als_score +
0.3 * item_score +
0.2 * content_score )

# implementation
def compute_final_score(row):
    return (
        0.5 * row["als_score"] +
        0.3 * row["item_score"] +
        0.2 * row["content_score"]
    )


NameError: name 'als_score' is not defined

STEP 5: BUSINESS RULE FILTERING
❓ Why business rules?

Because:

Models optimize math

Business optimizes revenue + trust

In [ ]:
def apply_business_rules(recs, user_id):
  purchased = set(
    final_data2[final_data2["user_id"] == user_id]["product_id"]
  )

  recs = recs[~recs["product_id"].isin(purchased)]
  recs = recs[recs["avg_review_score"] >= 3.5]

  return recs


STEP 6: DIVERSITY & EXPLORATION
❓ Why?

Avoid:

Same category spam

Filter bubbles

In [ ]:
def diversify(recs, max_per_category=2):
  return (
    recs.groupby("category").head(max_per_category)
  )

In [ ]:
# final top-K selection
final_recommendations = (
  ranked_recs
  .sort_values("final_score", ascending = False)
  .head(10)
)

NameError: name 'ranked_recs' is not defined

## PHASE 6: EVALUATION (OFFLINE + ONLINE)
 WHY EVALUATION IS HARD FOR RECOMMENDERS

Unlike classification:

No ground-truth labels

Multiple “correct” answers

User behavior is stochastic

So we use ranking-based metrics + business KPIs.

🔹 PART A: OFFLINE EVALUATION

(Before Deployment – Model Quality)

🔹 PART B: ONLINE EVALUATION

(After Deployment – Business Impact)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
import numpy as np
import pandas as pd
from tqdm import tqdm

print("--- Step 1: Split Interactions into Train/Test Sets ---")
# Use final_data2 which contains cleaned user-product interactions and scores
df_eval = final_data2[['user_id', 'product_id', 'interaction_score']].copy()

# Re-encode users and products to ensure contiguous indexing for sparse matrix
eval_user_encoder = LabelEncoder()
eval_product_encoder = LabelEncoder()
df_eval['user_idx'] = eval_user_encoder.fit_transform(df_eval['user_id'])
df_eval['product_idx'] = eval_product_encoder.fit_transform(df_eval['product_id'])

# Random 80/20 train/test split on interactions
train_df, test_df = train_test_split(df_eval, test_size=0.2, random_state=42)

# Create training sparse matrix
num_users = df_eval['user_idx'].max() + 1
num_products = df_eval['product_idx'].max() + 1

train_matrix = csr_matrix(
    (train_df['interaction_score'], (train_df['user_idx'], train_df['product_idx'])),
    shape=(num_users, num_products)
)

print("--- Step 2: Fit Evaluation Models ---")
# 1. Popularity Baseline: Sum interaction scores on train_df
popularity_scores = train_df.groupby('product_idx')['interaction_score'].sum().sort_values(ascending=False)
popular_items = popularity_scores.index.tolist()

# 2. Collaborative Filtering (ALS) Model
als_eval_model = AlternatingLeastSquares(factors=100, regularization=0.1, iterations=20, random_state=42)
als_eval_model.fit(train_matrix)

print("--- Step 3: Define Evaluation Metrics ---")
def precision_recall_at_k(recommended, actual, k=10):
    if not actual:
        return 0.0, 0.0
    recommended_k = recommended[:k]
    intersection = set(recommended_k).intersection(set(actual))
    precision = len(intersection) / k
    recall = len(intersection) / len(actual)
    return precision, recall

def map_at_k(recommended, actual, k=10):
    if not actual:
        return 0.0
    recommended_k = recommended[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(recommended_k):
        if p in actual:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score / min(len(actual), k)

# Group test set interactions by user index
test_truth = test_df.groupby('user_idx')['product_idx'].apply(set).to_dict()

# Only evaluate users with interactions in both train (to recommend) and test (to evaluate)
eval_users = [u for u in test_truth.keys() if train_matrix[u].nnz > 0]

# Sample 1,000 users to keep evaluation runtime fast (<15 seconds)
sample_size = min(1000, len(eval_users))
np.random.seed(42)
sampled_eval_users = np.random.choice(eval_users, size=sample_size, replace=False)

print(f"Evaluating models on a sample of {sample_size} test users...")

pop_precisions, pop_recalls, pop_maps = [], [], []
als_precisions, als_recalls, als_maps = [], [], []

for user_idx in tqdm(sampled_eval_users):
    actual_items = test_truth[user_idx]
    
    # 1. Popularity Recommender (Exclude items they already bought in the training set)
    user_train_items = set(train_df[train_df['user_idx'] == user_idx]['product_idx'])
    pop_recs = [item for item in popular_items if item not in user_train_items][:10]
    
    p_pop, r_pop = precision_recall_at_k(pop_recs, actual_items, k=10)
    map_pop = map_at_k(pop_recs, actual_items, k=10)
    pop_precisions.append(p_pop)
    pop_recalls.append(r_pop)
    pop_maps.append(map_pop)
    
    # 2. ALS Recommender
    ids, _ = als_eval_model.recommend(
        user_idx,
        train_matrix[user_idx],
        N=10
    )
    als_recs = list(ids)
    
    p_als, r_als = precision_recall_at_k(als_recs, actual_items, k=10)
    map_als = map_at_k(als_recs, actual_items, k=10)
    als_precisions.append(p_als)
    als_recalls.append(r_als)
    als_maps.append(map_als)

results = pd.DataFrame({
    'Metric': ['Precision@10', 'Recall@10', 'MAP@10'],
    'Popularity Recommender': [np.mean(pop_precisions), np.mean(pop_recalls), np.mean(pop_maps)],
    'ALS Collaborative Filtering': [np.mean(als_precisions), np.mean(als_recalls), np.mean(als_maps)]
})

print("\n=== OFFLINE EVALUATION RESULTS ===")
print(results.to_string(index=False))
